# 🛠️ LLM Tools Tutorial — Complete Guide
### From built-in LangChain tools to custom tool design, all via a local model
#
---
#
**What you'll learn:**
- ✅ **Built-in LangChain tools** — Wikipedia, DuckDuckGo, Python REPL, ArXiv, etc. (all free, no API keys)
- ✅ **Custom function tools** — `@tool` decorator, `StructuredTool`, `BaseTool` class
- ✅ **Tool parsing** — same robust `<tool_call>` text-mode parsing as the project builder
- ✅ **Agent loop** — connecting tools to the model in a ReAct-style loop
- ✅ **Advanced patterns** — tool chaining, conditional tools, tool with memory

## 1. Install Dependencies

In [1]:
!pip install openai langchain langchain-community langchain-core \
             duckduckgo-search wikipedia arxiv \
             rich tenacity json-repair -q

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.4/501.4 kB 30.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.0 MB/s eta 0:00:00


In [3]:
!pip install langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 5.3 MB/s eta 0:00:00


## 2. Imports & Custom OpenAI Client

In [4]:
import os, json, re, math, random, hashlib, datetime, traceback
from typing import Any, Optional

try:
    from json_repair import repair_json
    HAS_JSON_REPAIR = True
except ImportError:
    HAS_JSON_REPAIR = False

from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.syntax import Syntax

console = Console()

# ── Custom backend — plain chat, no tool API ────────────────────────────────
client = OpenAI(
    base_url="https://bimetallistic-noncuriously-lawanna.ngrok-free.dev/v1",
    api_key="dummy-key"
)

try:
    _models   = client.models.list()
    model_ids = [m.id for m in _models.data]
    MODEL     = model_ids[0]
    console.print(f"✅ Connected. Models: {model_ids}")
    console.print(f"   Using: [bold cyan]{MODEL}[/bold cyan]")
except Exception as e:
    MODEL = "gpt-oss-20b"
    console.print(f"[yellow]⚠️  Model list failed ({e}). Using MODEL='{MODEL}'[/yellow]")

MAX_TOKENS  = 4096
TEMPERATURE = 0.1
console.print(f"MODEL={MODEL}  MAX_TOKENS={MAX_TOKENS}  TEMP={TEMPERATURE}")
console.print("[bold green]Mode: pure text prompting — no tool API sent[/bold green]")

✅ Connected. Models: ['gpt-oss-20b']

Using: gpt-oss-20b

MODEL=gpt-oss-20b  MAX_TOKENS=4096  TEMP=0.1

Mode: pure text prompting — no tool API sent

## 3. Tool-Call Parser (Robust — same as Project Builder v4)
#
The model emits `<tool_call>{"name": ..., "arguments": {...}}</tool_call>` tags.
This parser handles:
- **Strict**: well-formed open+close tag
- **Loose**: missing closing tag / truncated JSON
- **Repaired**: malformed JSON fixed via `json_repair`

In [5]:
_TC_STRICT_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
_TC_OPEN_RE   = re.compile(r'<tool_call>\s*(\{.*?)(?=<tool_call>|</tool_call>|$)', re.DOTALL)


def _try_parse_json(raw: str) -> dict | None:
    raw = raw.strip()
    if not raw:
        return None
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    if HAS_JSON_REPAIR:
        try:
            repaired = repair_json(raw)
            if repaired:
                return json.loads(repaired)
        except Exception:
            pass
    # Manual bracket completion
    try:
        depth_curly = depth_square = 0
        in_string = escape_next = False
        last_good = 0
        for i, ch in enumerate(raw):
            if escape_next: escape_next = False; continue
            if ch == '\\' and in_string: escape_next = True; continue
            if ch == '"': in_string = not in_string; last_good = i + 1 if not in_string else last_good; continue
            if in_string: continue
            if ch in ('{', '['):
                depth_curly  += ch == '{'
                depth_square += ch == '['
            elif ch in ('}', ']'):
                depth_curly  -= ch == '}'
                depth_square -= ch == ']'
                if depth_curly >= 0 and depth_square >= 0:
                    last_good = i + 1
        candidate = raw[:last_good].rstrip().rstrip(',') if last_good else raw
        candidate += ']' * max(0, depth_square) + '}' * max(0, depth_curly)
        return json.loads(candidate)
    except Exception:
        pass
    return None


class _FakeFn:
    def __init__(self, name, args):
        self.name      = name
        self.arguments = json.dumps(args)

class _FakeTC:
    _n = 0
    def __init__(self, name, args, malformed: bool = False):
        _FakeTC._n   += 1
        self.id       = f"txt_{_FakeTC._n}"
        self.function = _FakeFn(name, args)
        self.malformed = malformed


def parse_text_tool_calls(text: str, tool_map: dict) -> tuple[list, bool]:
    """Returns (tool_calls, attempted)."""
    out = []
    attempted = '<tool_call>' in text

    for m in _TC_STRICT_RE.finditer(text):
        obj = _try_parse_json(m.group(1))
        if obj:
            name = obj.get("name") or obj.get("tool") or obj.get("function")
            args = obj.get("arguments") or obj.get("args") or obj.get("parameters") or {}
            if isinstance(args, str):
                try: args = json.loads(args)
                except: args = {}
            if name and name in tool_map:
                out.append(_FakeTC(name, args))
    if out:
        return out, attempted

    for m in _TC_OPEN_RE.finditer(text):
        obj = _try_parse_json(m.group(1))
        if obj:
            name = obj.get("name") or obj.get("tool") or obj.get("function")
            args = obj.get("arguments") or obj.get("args") or obj.get("parameters") or {}
            if isinstance(args, str):
                try: args = json.loads(args)
                except: args = {}
            if name and name in tool_map:
                console.print(f"  [yellow]🔧 Repaired truncated <tool_call> for '{name}'[/yellow]")
                out.append(_FakeTC(name, args, malformed=True))

    return out, attempted


print(f"✅ Parser ready. json_repair={HAS_JSON_REPAIR}")

✅ Parser ready. json_repair=True


## 4. LLM Caller — plain chat, no tool API

In [6]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=15))
def call_llm(messages: list, verbose_raw: bool = False, max_tokens: int = None):
    """
    Plain chat completion — no `tools`, no `tool_choice`.
    The model is instructed via system prompt to emit <tool_call> tags.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens or MAX_TOKENS,
        temperature=TEMPERATURE,
        # ── intentionally NO tools / tool_choice ──────────────────────────
    )
    if verbose_raw:
        ch = resp.choices[0]
        console.print(
            f"  [dim]► finish={ch.finish_reason}  "
            f"content_chars={len(ch.message.content or '')}[/dim]"
        )
    return resp


print("✅ LLM caller ready (plain chat — no tool API).")

✅ LLM caller ready (plain chat — no tool API).


## 5. Generic Agent Loop
#
A reusable single-turn agent that:
1. Sends the system prompt + user message
2. Parses `<tool_call>` from the response
3. Executes the tool and injects the result
4. Repeats until the model stops calling tools

In [7]:
def run_agent(
    user_query: str,
    system_prompt: str,
    tool_map: dict,
    max_iterations: int = 10,
    verbose: bool = True,
) -> dict:
    """
    Generic ReAct-style agent loop.
    Returns {"answer": str, "tool_calls": list, "iterations": int}
    """
    messages = [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": user_query},
    ]
    log = []
    final_answer = ""

    for it in range(1, max_iterations + 1):
        if verbose:
            console.print(f"\n[bold dim]── Iteration {it}/{max_iterations} ──[/bold dim]")

        try:
            resp = call_llm(messages, verbose_raw=verbose)
        except Exception as e:
            console.print(f"[red]LLM error: {e}[/red]")
            break

        text   = resp.choices[0].message.content or ""
        finish = resp.choices[0].finish_reason
        messages.append({"role": "assistant", "content": text})

        tool_calls, attempted = parse_text_tool_calls(text, tool_map)

        if not tool_calls:
            final_answer = text
            if verbose:
                console.print(Panel(text[:600], title="[green]Final Answer[/green]", expand=False))
            break

        for tc in tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments)
            except:
                args = {}

            if verbose:
                sa = {k: (str(v)[:60] + "…" if len(str(v)) > 60 else v) for k, v in args.items()}
                console.print(f"  🔧 [bold]{name}[/bold]({sa})")

            try:
                result = tool_map[name](**args)
            except Exception as ex:
                result = {"error": traceback.format_exc()[-400:]}

            result_str = json.dumps(result, indent=2) if isinstance(result, dict) else str(result)
            if verbose:
                console.print(Panel(result_str[:500], title=f"[cyan]Tool result: {name}[/cyan]", expand=False))

            log.append({"tool": name, "args": args, "result": result_str[:200]})
            messages.append({"role": "user", "content": f"Tool result for {name}:\n{result_str}"})

    return {"answer": final_answer, "tool_calls": log, "iterations": it}


print("✅ Generic agent loop ready.")

✅ Generic agent loop ready.


---
# 🧰 Part A — Built-in LangChain Tools (Free, No API Keys)
#
LangChain provides many ready-made tools. We wrap them so the agent can call them
via `<tool_call>` text tags.

## A1. Wikipedia Tool

In [8]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ── Instantiate LangChain built-in ─────────────────────────────────────────
_wiki_lc = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500))

# ── Wrap for our tool_map ──────────────────────────────────────────────────
def wikipedia_search(query: str) -> dict:
    """Search Wikipedia and return a short summary."""
    try:
        result = _wiki_lc.run(query)
        return {"status": "ok", "query": query, "result": result[:1000]}
    except Exception as e:
        return {"status": "error", "error": str(e)}


# ── Quick smoke test ───────────────────────────────────────────────────────
r = wikipedia_search("Python programming language")
console.print(Panel(r["result"][:400], title="[green]Wikipedia: Python[/green]", expand=False))
print("✅ Wikipedia tool ready.")

╭─────────────────────────────────────────────── Wikipedia: Python ───────────────────────────────────────────────╮
│ Page: Python (programming language)                                                                             │
│ Summary: Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code    │
│ readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected.  │
│ It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and │
│ functional programming.                                                                                         │
│ Guido va                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Wikipedia tool ready.


## A2. DuckDuckGo Search Tool

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

_ddg_lc = DuckDuckGoSearchRun()

def web_search(query: str) -> dict:
    """Search the web using DuckDuckGo (no API key required)."""
    try:
        result = _ddg_lc.run(query)
        return {"status": "ok", "query": query, "result": result[:1200]}
    except Exception as e:
        return {"status": "error", "error": str(e)}


r = web_search("latest Python version 2024")
console.print(Panel(r.get("result", r.get("error", ""))[:400], title="[green]DuckDuckGo[/green]", expand=False))
print("✅ DuckDuckGo tool ready.")

Impersonate 'safari_17.0' does not exist, using 'random'


╭────────────────────────────────────────────────── DuckDuckGo ───────────────────────────────────────────────────╮
│ Looking for Python with a different OS? Python for Windows, Linux/Unix, macOS, Android, other Want to help test │
│ development versions of Python 3.15? Pre-releases, Docker images List releases of Python , end of life, end of  │
│ support, support status, release date of each version , LTS versions . Python is a popular programming          │
│ langua... Want to discover Python version history? This blog post unveils                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ DuckDuckGo tool ready.


## A3. ArXiv Tool — Search Research Papers

In [10]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

_arxiv_lc = ArxivQueryRun(api_wrapper=ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=800))

def arxiv_search(query: str) -> dict:
    """Search ArXiv for academic papers."""
    try:
        result = _arxiv_lc.run(query)
        return {"status": "ok", "query": query, "result": result[:1200]}
    except Exception as e:
        return {"status": "error", "error": str(e)}


r = arxiv_search("retrieval augmented generation")
console.print(Panel(r.get("result", r.get("error", ""))[:500], title="[green]ArXiv: RAG[/green]", expand=False))
print("✅ ArXiv tool ready.")

╭────────────────────────────────────────────────── ArXiv: RAG ───────────────────────────────────────────────────╮
│ Published: 2025-06-14                                                                                           │
│ Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation                                       │
│ Authors: Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang                                                        │
│ Summary: We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image      │
│ generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior      │
│ methods that perform a single, static retrieval before generation and condition the entire generation on fixed  │
│ reference images, AR-R                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ ArXiv tool ready.


## A4. Python REPL Tool — Execute Code Dynamically
#
⚠️ **Use with caution in production!** Safe for controlled demos.

In [11]:
from langchain_experimental.tools import PythonREPLTool

_repl_lc = PythonREPLTool()

def python_repl(code: str) -> dict:
    """Execute Python code and return stdout. Use for math, data manipulation, etc."""
    try:
        output = _repl_lc.run(code)
        return {"status": "ok", "output": str(output)[:800]}
    except Exception as e:
        return {"status": "error", "error": str(e)}


r = python_repl("import math; print([math.factorial(n) for n in range(1,8)])")
console.print(Panel(r.get("output", r.get("error", "")), title="[green]Python REPL[/green]", expand=False))
print("✅ Python REPL tool ready.")

Python REPL can execute arbitrary code. Use with caution.


╭───────── Python REPL ─────────╮
│ [1, 2, 6, 24, 120, 720, 5040] │
│                               │
╰───────────────────────────────╯

✅ Python REPL tool ready.


## A5. Register All Built-in Tools

In [12]:
BUILTIN_TOOLS = {
    "wikipedia_search": wikipedia_search,
    "web_search":       web_search,
    "arxiv_search":     arxiv_search,
    "python_repl":      python_repl,
}

t = Table(title="Built-in LangChain Tools", show_header=True, header_style="bold magenta")
t.add_column("Tool",        style="bold cyan")
t.add_column("Source")
t.add_column("API Key?")
t.add_column("Use case")
rows = [
    ("wikipedia_search", "LangChain / wikipedia",           "No",  "Factual lookups"),
    ("web_search",       "LangChain / duckduckgo-search",   "No",  "Current web info"),
    ("arxiv_search",     "LangChain / arxiv",               "No",  "Research papers"),
    ("python_repl",      "langchain-experimental",          "No",  "Computation / code"),
]
for row in rows:
    t.add_row(*row)
console.print(t)
print(f"\n✅ {len(BUILTIN_TOOLS)} built-in tools registered.")

                              Built-in LangChain Tools                              
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Tool             ┃ Source                        ┃ API Key? ┃ Use case           ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ wikipedia_search │ LangChain / wikipedia         │ No       │ Factual lookups    │
│ web_search       │ LangChain / duckduckgo-search │ No       │ Current web info   │
│ arxiv_search     │ LangChain / arxiv             │ No       │ Research papers    │
│ python_repl      │ langchain-experimental        │ No       │ Computation / code │
└──────────────────┴───────────────────────────────┴──────────┴────────────────────┘


✅ 4 built-in tools registered.


## A6. Demo — Agent Using Built-in Tools

In [13]:
_NAMES_BUILTIN = ", ".join(BUILTIN_TOOLS)

BUILTIN_SYSTEM = f"""You are a research assistant with access to tools.
Always use a tool to look up information rather than relying on memory.

════════════════════════════════════════════════════
OUTPUT FORMAT — MANDATORY
════════════════════════════════════════════════════
Emit ONE <tool_call> block per response. After all tools are done, write your final answer.

<tool_call>
{{"name": "tool_name", "arguments": {{"key": "value"}}}}
</tool_call>

AVAILABLE TOOLS: {_NAMES_BUILTIN}

wikipedia_search(query: str)  — Wikipedia summary
web_search(query: str)        — DuckDuckGo web search  
arxiv_search(query: str)      — ArXiv academic papers
python_repl(code: str)        — execute Python, get stdout
"""

console.rule("[bold cyan]Demo: Built-in Tools Agent[/bold cyan]")
result_builtin = run_agent(
    user_query="Who invented the transformer architecture and which institution were they from? Also compute 2^32.",
    system_prompt=BUILTIN_SYSTEM,
    tool_map=BUILTIN_TOOLS,
    verbose=True,
)

─────────────────────────────────────────── Demo: Built-in Tools Agent ────────────────────────────────────────────

── Iteration 1/10 ──

► finish=stop  content_chars=110

🔧 wikipedia_search({'query': 'transformer architecture invented'})

╭───────────────────────────────────────── Tool result: wikipedia_search ─────────────────────────────────────────╮
│ {                                                                                                               │
│   "status": "ok",                                                                                               │
│   "query": "transformer architecture invented",                                                                 │
│   "result": "Page: Transformer (deep learning)\nSummary: In deep learning, the transformer is an artificial     │
│ neural network architecture based on the multi-head attention mechanism, in which text is converted to          │
│ numerical representations called tokens, and each token is converted into a vector via lookup from a word       │
│ embedding table. At each layer, each token is then contextualized within the scope of the context window with   │
│ other                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

── Iteration 2/10 ──

► finish=stop  content_chars=326

╭────────────────────────────────────────── Final Answer ───────────────────────────────────────────╮
│ The transformer architecture was introduced in the 2017 paper **“Attention Is All You Need.”**    │
│ The authors were researchers from **Google Brain** (and Google Research).                         │
│                                                                                                   │
│ Now, compute \(2^{32}\):                                                                          │
│                                                                                                   │
│ [                                                                                                 │
│ 2^{32}=4\,294\,967\,296                                                                           │
│ \]                                                                                                │
│                                                                                                   │
│ So, the transformer was invented by Google Brain researchers, and \(2^{32}=4{,}294{,}967{,}296\). │
╰───────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# 🔨 Part B — Custom Tools (3 Patterns)
#
| Pattern | When to use |
|---------|------------|
| Plain function | Simple, stateless |
| LangChain `@tool` decorator | Auto-generates schema from docstring |
| `BaseTool` subclass | Stateful, complex validation, async |

## B1. Pattern 1 — Plain Python Functions
#
The simplest approach: just write a function and put it in `tool_map`.

In [14]:
# ── Custom Tool 1: Unit Converter ────────────────────────────────────────────
def unit_convert(value: float, from_unit: str, to_unit: str) -> dict:
    """
    Convert between common units.
    Supported: km↔miles, kg↔lbs, celsius↔fahrenheit, liters↔gallons
    """
    key = (from_unit.lower(), to_unit.lower())
    conversions = {
        ("km",         "miles"):      lambda v: v * 0.621371,
        ("miles",      "km"):          lambda v: v * 1.60934,
        ("kg",         "lbs"):         lambda v: v * 2.20462,
        ("lbs",        "kg"):          lambda v: v * 0.453592,
        ("celsius",    "fahrenheit"):  lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"):     lambda v: (v - 32) * 5/9,
        ("liters",     "gallons"):     lambda v: v * 0.264172,
        ("gallons",    "liters"):      lambda v: v * 3.78541,
    }
    if key not in conversions:
        return {"status": "error", "error": f"Unknown conversion: {from_unit} → {to_unit}"}
    converted = round(conversions[key](value), 4)
    return {
        "status":     "ok",
        "input":      f"{value} {from_unit}",
        "output":     f"{converted} {to_unit}",
        "converted":  converted,
    }


# ── Custom Tool 2: Text Statistics ──────────────────────────────────────────
def text_stats(text: str) -> dict:
    """
    Compute reading statistics for a block of text.
    Returns word count, sentence count, average word length, estimated read time.
    """
    words     = text.split()
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chars     = sum(len(w) for w in words)
    return {
        "status":           "ok",
        "word_count":       len(words),
        "sentence_count":   len(sentences),
        "char_count":       len(text),
        "avg_word_length":  round(chars / len(words), 2) if words else 0,
        "read_time_secs":   round(len(words) / 238 * 60),   # avg 238 wpm
        "unique_words":     len(set(w.lower().strip('.,!?') for w in words)),
    }


# ── Custom Tool 3: Password Generator ──────────────────────────────────────
import secrets, string

def generate_password(
    length: int = 16,
    use_symbols: bool = True,
    use_digits: bool = True,
    use_uppercase: bool = True,
) -> dict:
    """Generate a cryptographically secure random password."""
    charset = string.ascii_lowercase
    if use_uppercase: charset += string.ascii_uppercase
    if use_digits:    charset += string.digits
    if use_symbols:   charset += "!@#$%^&*()-_=+"
    pw = "".join(secrets.choice(charset) for _ in range(length))
    entropy = math.log2(len(charset) ** length)
    return {
        "status":   "ok",
        "password": pw,
        "length":   length,
        "entropy_bits": round(entropy, 1),
        "strength": "strong" if entropy > 80 else "medium" if entropy > 50 else "weak",
    }


# ── Smoke tests ──────────────────────────────────────────────────────────────
console.print(unit_convert(100, "km", "miles"))
console.print(text_stats("The quick brown fox jumps over the lazy dog. It did so twice."))
console.print(generate_password(20))
print("✅ Plain function custom tools ready.")

{'status': 'ok', 'input': '100 km', 'output': '62.1371 miles', 'converted': 62.1371}

{
    'status': 'ok',
    'word_count': 13,
    'sentence_count': 2,
    'char_count': 61,
    'avg_word_length': 3.77,
    'read_time_secs': 3,
    'unique_words': 12
}

{'status': 'ok', 'password': 'itec++e16S@_!Y@kx_D!', 'length': 20, 'entropy_bits': 125.0, 'strength': 'strong'}

✅ Plain function custom tools ready.


## B2. Pattern 2 — LangChain `@tool` Decorator
#
LangChain reads the function signature + docstring to auto-generate a JSON schema.
We still call the function directly in our agent loop.

In [15]:
from langchain_core.tools import tool

@tool
def currency_exchange(amount: float, from_currency: str, to_currency: str) -> dict:
    """
    Convert between currencies using hardcoded approximate rates (demo only).
    Supports: USD, EUR, GBP, JPY, BDT, INR, CAD, AUD.
    Args:
        amount: The amount to convert.
        from_currency: Source currency code (e.g., 'USD').
        to_currency: Target currency code (e.g., 'EUR').
    """
    # Base rates relative to USD (approximate, for demo)
    rates = {
        "USD": 1.000, "EUR": 0.917, "GBP": 0.786, "JPY": 149.5,
        "BDT": 110.0, "INR": 83.2,  "CAD": 1.360, "AUD": 1.530,
    }
    fc, tc = from_currency.upper(), to_currency.upper()
    if fc not in rates or tc not in rates:
        return {"status": "error", "error": f"Unsupported currency. Available: {list(rates)}"}
    usd_amount = amount / rates[fc]
    result     = round(usd_amount * rates[tc], 4)
    return {
        "status":        "ok",
        "input":         f"{amount} {fc}",
        "output":        f"{result} {tc}",
        "exchange_rate": round(rates[tc] / rates[fc], 6),
    }


@tool
def date_calculator(date_str: str, operation: str, days: int = 0) -> dict:
    """
    Perform date arithmetic.
    Args:
        date_str: Start date in YYYY-MM-DD format (use 'today' for today).
        operation: One of 'add', 'subtract', 'weekday', 'diff_from_today'.
        days: Number of days to add/subtract (for add/subtract operations).
    """
    today = datetime.date.today()
    d = today if date_str.lower() == 'today' else datetime.date.fromisoformat(date_str)
    ops = {
        "add":            lambda: str(d + datetime.timedelta(days=days)),
        "subtract":       lambda: str(d - datetime.timedelta(days=days)),
        "weekday":        lambda: d.strftime("%A"),
        "diff_from_today": lambda: (today - d).days,
    }
    if operation not in ops:
        return {"status": "error", "error": f"Unknown operation. Choose: {list(ops)}"}
    return {"status": "ok", "input": str(d), "operation": operation, "result": ops[operation]()}


# LangChain @tool auto-generates a schema — inspect it
console.print("[bold]currency_exchange schema:[/bold]")
console.print(currency_exchange.args_schema.schema())

# Still callable as plain Python for our agent loop:
console.print(currency_exchange.invoke({"amount": 1000, "from_currency": "USD", "to_currency": "BDT"}))
console.print(date_calculator.invoke({"date_str": "today", "operation": "weekday"}))
print("✅ @tool decorator tools ready.")

currency_exchange schema:

/tmp/ipykernel_17/762948491.py:55: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  console.print(currency_exchange.args_schema.schema())


{
    'description': "Convert between currencies using hardcoded approximate rates (demo only).\nSupports: USD, EUR, 
GBP, JPY, BDT, INR, CAD, AUD.\nArgs:\n    amount: The amount to convert.\n    from_currency: Source currency code 
(e.g., 'USD').\n    to_currency: Target currency code (e.g., 'EUR').",
    'properties': {
        'amount': {'title': 'Amount', 'type': 'number'},
        'from_currency': {'title': 'From Currency', 'type': 'string'},
        'to_currency': {'title': 'To Currency', 'type': 'string'}
    },
    'required': ['amount', 'from_currency', 'to_currency'],
    'title': 'currency_exchange',
    'type': 'object'
}

{'status': 'ok', 'input': '1000.0 USD', 'output': '110000.0 BDT', 'exchange_rate': 110.0}

{'status': 'ok', 'input': '2026-02-20', 'operation': 'weekday', 'result': 'Friday'}

✅ @tool decorator tools ready.


## B3. Pattern 3 — `BaseTool` Subclass (Stateful / Complex)
#
Use `BaseTool` when you need:
- Internal state (e.g. a database connection, cache)
- Custom input validation
- Async support
- Detailed metadata (name, description, tags)

In [16]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field


# ── Pydantic input schema ───────────────────────────────────────────────────
class MemoInput(BaseModel):
    action:  str = Field(description="One of: 'write', 'read', 'list', 'delete'")
    key:     str = Field(default="", description="The memo key/name")
    content: str = Field(default="", description="Content to write (for 'write' action)")


class MemoTool(BaseTool):
    """
    Stateful in-memory key-value memo store.
    Demonstrates BaseTool with internal state and Pydantic validation.
    """
    name:        str = "memo"
    description: str = (
        "A persistent memo store. Actions: "
        "write(key, content), read(key), list(), delete(key). "
        "Use this to save and retrieve information across steps."
    )
    args_schema: type[BaseModel] = MemoInput

    # Internal state — persists across calls
    _store: dict = {}

    def _run(self, action: str, key: str = "", content: str = "") -> dict:
        if action == "write":
            if not key:
                return {"status": "error", "error": "key is required for write"}
            self._store[key] = content
            return {"status": "written", "key": key, "length": len(content)}

        elif action == "read":
            if key not in self._store:
                return {"status": "not_found", "key": key}
            return {"status": "ok", "key": key, "content": self._store[key]}

        elif action == "list":
            return {"status": "ok", "keys": list(self._store), "count": len(self._store)}

        elif action == "delete":
            removed = self._store.pop(key, None)
            return {"status": "deleted" if removed is not None else "not_found", "key": key}

        else:
            return {"status": "error", "error": f"Unknown action '{action}'"}

    async def _arun(self, **kwargs) -> dict:
        return self._run(**kwargs)


# ── Instantiate and create a plain-function wrapper for tool_map ─────────────
_memo_tool = MemoTool()

def memo(action: str, key: str = "", content: str = "") -> dict:
    """Wrapper so MemoTool works in our plain-function tool_map."""
    return _memo_tool._run(action=action, key=key, content=content)


# ── Smoke test ───────────────────────────────────────────────────────────────
console.print(memo("write",  key="goal",    content="Learn about LLM tools"))
console.print(memo("write",  key="note1",   content="BaseTool supports async"))
console.print(memo("read",   key="goal"))
console.print(memo("list"))
console.print(memo("delete", key="note1"))
print("✅ BaseTool (MemoTool) ready.")

{'status': 'written', 'key': 'goal', 'length': 21}

{'status': 'written', 'key': 'note1', 'length': 23}

{'status': 'ok', 'key': 'goal', 'content': 'Learn about LLM tools'}

{'status': 'ok', 'keys': ['goal', 'note1'], 'count': 2}

{'status': 'deleted', 'key': 'note1'}

✅ BaseTool (MemoTool) ready.


## B4. Register All Custom Tools

In [17]:
# Wrap @tool decorated functions: .invoke() → plain call
def _currency_exchange(**kwargs): return currency_exchange.invoke(kwargs)
def _date_calculator(**kwargs):   return date_calculator.invoke(kwargs)

CUSTOM_TOOLS = {
    # Pattern 1 — plain functions
    "unit_convert":       unit_convert,
    "text_stats":         text_stats,
    "generate_password":  generate_password,
    # Pattern 2 — @tool decorator (wrapped for tool_map)
    "currency_exchange":  _currency_exchange,
    "date_calculator":    _date_calculator,
    # Pattern 3 — BaseTool subclass (wrapped)
    "memo":               memo,
}

t = Table(title="Custom Tools", show_header=True, header_style="bold magenta")
t.add_column("Tool",    style="bold cyan")
t.add_column("Pattern")
t.add_column("Description")
rows = [
    ("unit_convert",      "Plain function",      "Temperature, length, weight, volume"),
    ("text_stats",        "Plain function",      "Word count, readability metrics"),
    ("generate_password", "Plain function",      "Crypto-secure passwords"),
    ("currency_exchange", "@tool decorator",     "Currency conversion with schema"),
    ("date_calculator",   "@tool decorator",     "Date arithmetic & weekday lookup"),
    ("memo",              "BaseTool subclass",   "Stateful KV memo store"),
]
for row in rows:
    t.add_row(*row)
console.print(t)
print(f"\n✅ {len(CUSTOM_TOOLS)} custom tools registered.")

                                 Custom Tools                                  
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tool              ┃ Pattern           ┃ Description                         ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ unit_convert      │ Plain function    │ Temperature, length, weight, volume │
│ text_stats        │ Plain function    │ Word count, readability metrics     │
│ generate_password │ Plain function    │ Crypto-secure passwords             │
│ currency_exchange │ @tool decorator   │ Currency conversion with schema     │
│ date_calculator   │ @tool decorator   │ Date arithmetic & weekday lookup    │
│ memo              │ BaseTool subclass │ Stateful KV memo store              │
└───────────────────┴───────────────────┴─────────────────────────────────────┘


✅ 6 custom tools registered.


## B5. Demo — Agent Using Custom Tools

In [18]:
_NAMES_CUSTOM = ", ".join(CUSTOM_TOOLS)

CUSTOM_SYSTEM = f"""You are a helpful assistant with access to utility tools.
Use the appropriate tool for each sub-task. Save important results with the memo tool.

════════════════════════════════════════════════════
OUTPUT FORMAT — MANDATORY
════════════════════════════════════════════════════
Emit ONE <tool_call> block per response, then wait for the result.
When all steps are done, write your final answer (no tool_call tag).

<tool_call>
{{"name": "tool_name", "arguments": {{"key": "value"}}}}
</tool_call>

AVAILABLE TOOLS: {_NAMES_CUSTOM}

unit_convert(value, from_unit, to_unit)         — unit conversion
text_stats(text)                                — readability stats
generate_password(length, use_symbols, ...)     — secure password
currency_exchange(amount, from_currency, to_currency) — FX rates
date_calculator(date_str, operation, days)      — date arithmetic
memo(action, key, content)                      — read/write notes
"""

console.rule("[bold cyan]Demo: Custom Tools Agent[/bold cyan]")
result_custom = run_agent(
    user_query=(
        "I ran a 10km race. How far is that in miles? "
        "Also convert 25 celsius to fahrenheit and save both results as memos. "
        "Finally, generate a 20-character password."
    ),
    system_prompt=CUSTOM_SYSTEM,
    tool_map=CUSTOM_TOOLS,
    verbose=True,
)

──────────────────────────────────────────── Demo: Custom Tools Agent ─────────────────────────────────────────────

── Iteration 1/10 ──

► finish=stop  content_chars=107

🔧 unit_convert({'value': '10', 'from_unit': 'km', 'to_unit': 'mi'})

╭────────── Tool result: unit_convert ──────────╮
│ {                                             │
│   "status": "error",                          │
│   "error": "Unknown conversion: km \u2192 mi" │
│ }                                             │
╰───────────────────────────────────────────────╯

── Iteration 2/10 ──

► finish=stop  content_chars=356

╭───────────────────────────────────────────────── Final Answer ──────────────────────────────────────────────────╮
│ It looks like the unit conversion tool doesn’t support converting kilometers to miles directly. I’ll do the     │
│ conversion manually: 1 km ≈ 0.621371 mi, so 10 km ≈ 6.21371 mi. I’ll store that result as a memo.               │
│                                                                                                                 │
│ Next, I’ll convert 25 °C to °F: °F = °C × 9/5 + 32, so 25 °C ≈ 77 °F. I’ll store that as a memo too.            │
│                                                                                                                 │
│ Finally, I’ll generate a 20‑character password.                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# 🔗 Part C — Advanced Patterns

## C1. Combined Tool Set — Built-in + Custom
#
Combine all tools into one mega-registry. The agent picks the right one.

In [19]:
ALL_TOOLS = {**BUILTIN_TOOLS, **CUSTOM_TOOLS}
_NAMES_ALL = ", ".join(ALL_TOOLS)

ALL_TOOLS_SYSTEM = f"""You are an expert research and utility assistant.
You have access to search, computation, and utility tools.
Always use tools to get accurate information. Chain multiple tools as needed.
Save key findings with the memo tool before giving your final answer.

════════════════════════════════════════════════════
OUTPUT FORMAT — MANDATORY
════════════════════════════════════════════════════
Emit exactly ONE <tool_call> per response. After results, decide next step.

<tool_call>
{{"name": "tool_name", "arguments": {{"key": "value"}}}}
</tool_call>

TOOLS ({len(ALL_TOOLS)} total): {_NAMES_ALL}

SEARCH:   wikipedia_search(query), web_search(query), arxiv_search(query)
CODE:     python_repl(code)
UTILS:    unit_convert(value,from_unit,to_unit), text_stats(text)
          currency_exchange(amount,from_currency,to_currency)
          date_calculator(date_str,operation,days)
          generate_password(length,use_symbols,use_digits,use_uppercase)
MEMORY:   memo(action,key,content)  — actions: write|read|list|delete
"""

console.rule("[bold cyan]Demo: Combined Tools — Research + Computation[/bold cyan]")
result_combined = run_agent(
    user_query=(
        "Look up what GPT-4 is on Wikipedia, then use the Python REPL to "
        "count how many words are in the summary you found. "
        "Save both the summary and word count to memos."
    ),
    system_prompt=ALL_TOOLS_SYSTEM,
    tool_map=ALL_TOOLS,
    verbose=True,
)

────────────────────────────────── Demo: Combined Tools — Research + Computation ──────────────────────────────────

── Iteration 1/10 ──

► finish=stop  content_chars=82

🔧 wikipedia_search({'query': 'GPT-4'})

╭───────────────────────────────────────── Tool result: wikipedia_search ─────────────────────────────────────────╮
│ {                                                                                                               │
│   "status": "ok",                                                                                               │
│   "query": "GPT-4",                                                                                             │
│   "result": "Page: GPT-4\nSummary: Generative Pre-trained Transformer 4 (GPT-4) is a large language model       │
│ developed by OpenAI and the fourth in its series of GPT foundation models.  \nGPT-4 is more capable than its    │
│ predecessor GPT-3.5 and followed by its successor GPT-5. GPT-4V is a version of GPT-4 that can process images   │
│ in addition to text. OpenAI has not revealed technical details and statistics about GPT-4, such as the precise  │
│ size of the model.\nAn e                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

🔧 memo({'action': 'write', 'key': 'gpt4_word_count', 'content': '61'})

╭───── Tool result: memo ─────╮
│ {                           │
│   "status": "written",      │
│   "key": "gpt4_word_count", │
│   "length": 2               │
│ }                           │
╰─────────────────────────────╯

── Iteration 5/10 ──

► finish=stop  content_chars=168

╭───────────────────────────────────────────────── Final Answer ──────────────────────────────────────────────────╮
│ The summary of GPT‑4 from Wikipedia has been stored in the memo key **gpt4_summary**, and the word count (61    │
│ words) has been stored in the memo key **gpt4_word_count**.                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## C2. Tool Chaining Pattern — Multi-step Research Pipeline
#
The agent is prompted to execute a specific sequence:
1. Search → 2. Extract → 3. Compute → 4. Summarize → 5. Save

In [20]:
CHAIN_SYSTEM = f"""You are a systematic research analyst. For every query:
STEP 1: wikipedia_search to get background
STEP 2: arxiv_search to find recent papers  
STEP 3: python_repl to do any computation (e.g. count papers, parse dates)
STEP 4: memo(write) to save your synthesis
STEP 5: Write final structured answer

Follow these steps in order. Use one <tool_call> per step.

<tool_call>
{{"name": "tool_name", "arguments": {{"key": "value"}}}}
</tool_call>

TOOLS: wikipedia_search, arxiv_search, python_repl, memo

wikipedia_search(query: str)
arxiv_search(query: str)
python_repl(code: str)
memo(action: str, key: str, content: str)
"""

CHAIN_TOOLS = {
    "wikipedia_search": wikipedia_search,
    "arxiv_search":     arxiv_search,
    "python_repl":      python_repl,
    "memo":             memo,
}

console.rule("[bold cyan]Demo: Tool Chaining Pipeline[/bold cyan]")
result_chain = run_agent(
    user_query="Research 'reinforcement learning from human feedback (RLHF)' — background, papers, and a brief synthesis.",
    system_prompt=CHAIN_SYSTEM,
    tool_map=CHAIN_TOOLS,
    max_iterations=8,
    verbose=True,
)

────────────────────────────────────────── Demo: Tool Chaining Pipeline ───────────────────────────────────────────

── Iteration 1/8 ──

► finish=stop  content_chars=16751

╭───────────────────────────────────────────────── Final Answer ──────────────────────────────────────────────────╮
│ **Step 1 – Wikipedia search**                                                                                   │
│ <assistant to=python_repl code>import wikipedia, json, sys, textwrap, re, math, random, statistics, fractions,  │
│ decimal, fractions, itertools, collections, functools, datetime, time, os, pathlib, typing, math, statistics,   │
│ fractions, decimal, itertools, collections, functools, datetime, time, os, pathlib, typing, json, sys, re,      │
│ textwrap, math, random, statistics, fractions, decimal, itertools, collections, functools, datetime, time, os,  │
│ pathlib, typing, json, sys, re, textwrap, math, random, statistics, fractions, decimal, itertools, collections, │
│ functools, dateti                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## C3. Conditional Tool — Validate Before Acting
#
A custom tool that validates its input and selectively delegates,
demonstrating how to build guard-rails into tools themselves.

In [21]:
def smart_lookup(query: str, mode: str = "auto") -> dict:
    """
    Conditional dispatcher: routes to wikipedia, arxiv, or web_search
    based on the query or explicit mode override.

    mode: 'auto' | 'wiki' | 'paper' | 'web'
    - auto: detects keywords like 'paper', 'research', 'study' → arxiv
            detects 'who', 'what is', 'define' → wikipedia
            otherwise → web_search
    """
    q = query.lower()

    if mode == "auto":
        if any(kw in q for kw in ["paper", "research", "study", "arxiv", "survey", "dataset"]):
            mode = "paper"
        elif any(kw in q for kw in ["who is", "what is", "define", "history of", "invented"]):
            mode = "wiki"
        else:
            mode = "web"

    dispatch = {
        "wiki":  lambda: wikipedia_search(query),
        "paper": lambda: arxiv_search(query),
        "web":   lambda: web_search(query),
    }

    if mode not in dispatch:
        return {"status": "error", "error": f"Unknown mode '{mode}'"}

    result = dispatch[mode]()
    result["routed_to"] = mode
    return result


# ── Smoke tests ───────────────────────────────────────────────────────────────
console.print("[bold]auto → wiki:[/bold]",  smart_lookup("Who is Alan Turing")["routed_to"])
console.print("[bold]auto → paper:[/bold]", smart_lookup("latest research on diffusion models")["routed_to"])
console.print("[bold]auto → web:[/bold]",   smart_lookup("best Python IDE 2024")["routed_to"])

CONDITIONAL_TOOLS = {
    "smart_lookup": smart_lookup,
    "python_repl":  python_repl,
    "memo":         memo,
}

COND_SYSTEM = """You are a smart research agent. Use smart_lookup for ALL lookups.
It automatically routes to Wikipedia, ArXiv, or web search.
For follow-up computation, use python_repl. Save conclusions with memo.

<tool_call>
{"name": "tool_name", "arguments": {"key": "value"}}
</tool_call>

TOOLS: smart_lookup(query, mode='auto'), python_repl(code), memo(action, key, content)
"""

console.rule("[bold cyan]Demo: Conditional Smart Lookup[/bold cyan]")
result_cond = run_agent(
    user_query="What is attention mechanism in neural networks? Also find a recent paper on it.",
    system_prompt=COND_SYSTEM,
    tool_map=CONDITIONAL_TOOLS,
    verbose=True,
)
print("✅ Conditional tool pattern demonstrated.")

auto → wiki: wiki

auto → paper: paper

auto → web: web

───────────────────────────────────────── Demo: Conditional Smart Lookup ──────────────────────────────────────────

── Iteration 1/10 ──

► finish=stop  content_chars=2083

╭───────────────────────────────────────────────── Final Answer ──────────────────────────────────────────────────╮
│ **Attention mechanism in neural networks**                                                                      │
│                                                                                                                 │
│ The attention mechanism is a technique that allows a neural network to focus on specific parts of its input     │
│ when producing an output.                                                                                       │
│ Instead of treating all input elements equally, the network learns *attention weights* that indicate how much   │
│ each element should contribute to the current computation.                                                      │
│                                                                                                                 │
│ Typical steps in an attention layer:                                                                            │
│                                                                                                                 │
│ 1. **Query, Key, Value** – For each element in the input sequence, three vectors are computed:                  │
│    *Query* (q) – what the model is looking for.                                                                 │
│    *Key* (k) – what each input element contains.                                                                │
│    *Value* (                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Conditional tool pattern demonstrated.


## C4. Tool with Memory — Accumulating Context Across Calls
#
A `BaseTool` that maintains a running conversation summary across multiple
tool invocations, useful for long research sessions.

In [22]:
class ResearchLogTool(BaseTool):
    """
    A stateful research log that accumulates findings.
    Actions: 'append', 'read', 'summarize', 'clear'.
    """
    name:        str = "research_log"
    description: str = (
        "Maintain a running research log. "
        "Actions: append(note), read(), summarize(), clear()."
    )
    args_schema: type[BaseModel] = None  # accept free-form

    _log: list = []

    def _run(self, action: str, note: str = "") -> dict:
        if action == "append":
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            self._log.append(f"[{ts}] {note}")
            return {"status": "appended", "total_entries": len(self._log)}

        elif action == "read":
            return {"status": "ok", "log": self._log, "count": len(self._log)}

        elif action == "summarize":
            if not self._log:
                return {"status": "empty"}
            return {
                "status":  "ok",
                "entries": len(self._log),
                "first":   self._log[0],
                "last":    self._log[-1],
                "all":     "\n".join(self._log),
            }

        elif action == "clear":
            n = len(self._log)
            self._log.clear()
            return {"status": "cleared", "removed_entries": n}

        return {"status": "error", "error": f"Unknown action: {action}"}

    async def _arun(self, **kwargs):
        return self._run(**kwargs)


_rl_tool = ResearchLogTool()

def research_log(action: str, note: str = "") -> dict:
    return _rl_tool._run(action=action, note=note)


# ── Smoke test ────────────────────────────────────────────────────────────────
research_log("append", "Python was created by Guido van Rossum")
research_log("append", "First released in 1991")
console.print(research_log("summarize"))
print("✅ ResearchLogTool (BaseTool) ready.")

{
    'status': 'ok',
    'entries': 2,
    'first': '[05:15:00] Python was created by Guido van Rossum',
    'last': '[05:15:00] First released in 1991',
    'all': '[05:15:00] Python was created by Guido van Rossum\n[05:15:00] First released in 1991'
}

✅ ResearchLogTool (BaseTool) ready.


---
# 📋 Summary — All Tool Types at a Glance

In [23]:
t = Table(title="🛠️ LLM Tool Tutorial — Complete Tool Registry", show_header=True, header_style="bold magenta")
t.add_column("#",          style="dim",       width=3)
t.add_column("Tool",       style="bold cyan",  width=22)
t.add_column("Category",   style="yellow",     width=14)
t.add_column("Pattern",    style="green",      width=18)
t.add_column("API Key?",                       width=8)
t.add_column("Use Case")

all_tool_rows = [
    # Built-in
    ("1",  "wikipedia_search",   "Built-in",  "LangChain wrapper",    "No",  "Factual lookups"),
    ("2",  "web_search",         "Built-in",  "LangChain wrapper",    "No",  "Current web info"),
    ("3",  "arxiv_search",       "Built-in",  "LangChain wrapper",    "No",  "Research papers"),
    ("4",  "python_repl",        "Built-in",  "LangChain experimental","No", "Code execution"),
    # Custom — plain function
    ("5",  "unit_convert",       "Custom",    "Plain function",       "No",  "Unit conversion"),
    ("6",  "text_stats",         "Custom",    "Plain function",       "No",  "Readability stats"),
    ("7",  "generate_password",  "Custom",    "Plain function",       "No",  "Secure passwords"),
    # Custom — @tool decorator
    ("8",  "currency_exchange",  "Custom",    "@tool decorator",      "No",  "FX conversion"),
    ("9",  "date_calculator",    "Custom",    "@tool decorator",      "No",  "Date arithmetic"),
    # Custom — BaseTool
    ("10", "memo",               "Custom",    "BaseTool subclass",    "No",  "Stateful KV store"),
    ("11", "smart_lookup",       "Advanced",  "Conditional dispatch", "No",  "Auto-routing search"),
    ("12", "research_log",       "Advanced",  "BaseTool + state",     "No",  "Accumulating log"),
]

for row in all_tool_rows:
    t.add_row(*row)

console.print(t)
console.print(f"\n[bold]Total: {len(all_tool_rows)} tools across 3 patterns + 2 advanced[/bold]")
console.print("[dim]All tools use plain text <tool_call> format — no OpenAI tool API required.[/dim]")

                             🛠️ LLM Tool Tutorial — Complete Tool Registry                              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Tool                   ┃ Category       ┃ Pattern            ┃ API Key? ┃ Use Case            ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ wikipedia_search       │ Built-in       │ LangChain wrapper  │ No       │ Factual lookups     │
│ 2   │ web_search             │ Built-in       │ LangChain wrapper  │ No       │ Current web info    │
│ 3   │ arxiv_search           │ Built-in       │ LangChain wrapper  │ No       │ Research papers     │
│ 4   │ python_repl            │ Built-in       │ LangChain          │ No       │ Code execution      │
│     │                        │                │ experimental       │          │                     │
│ 5   │ unit_convert           │ Custom         │ Plain function     │ No       │ Unit conversion     │
│ 6   │ text_stats             │ Custom         │ Plain function     │ No       │ Readability stats   │
│ 7   │ generate_password      │ Custom         │ Plain function     │ No       │ Secure passwords    │
│ 8   │ currency_exchange      │ Custom         │ @tool decorator    │ No       │ FX conversion       │
│ 9   │ date_calculator        │ Custom         │ @tool decorator    │ No       │ Date arithmetic     │
│ 10  │ memo                   │ Custom         │ BaseTool subclass  │ No       │ Stateful KV store   │
│ 11  │ smart_lookup           │ Advanced       │ Conditional        │ No       │ Auto-routing search │
│     │                        │                │ dispatch           │          │                     │
│ 12  │ research_log           │ Advanced       │ BaseTool + state   │ No       │ Accumulating log    │
└─────┴────────────────────────┴────────────────┴────────────────────┴──────────┴─────────────────────┘

Total: 12 tools across 3 patterns + 2 advanced

All tools use plain text <tool_call> format — no OpenAI tool API required.

---
## Troubleshooting Guide
#
| Symptom | Root Cause | Fix |
|---------|-----------|-----|
| `ModuleNotFoundError: langchain_community` | Package not installed | `!pip install langchain-community` |
| `ModuleNotFoundError: langchain_experimental` | REPL tool missing | `!pip install langchain-experimental` |
| DuckDuckGo returns empty | Rate limit / network | Retry after a few seconds; reduce query frequency |
| `<tool_call>` emitted, no closing tag | Model output truncated | Loose parser handles this; if persistent, raise `MAX_TOKENS` |
| `tool not in tool_map` | Wrong tool name in call | Check tool name spelling matches `tool_map` keys |
| `context_resets` / no-tool loop | Model not following format | Simplify system prompt; use worked example |
| `BaseTool._store` not persisting | Created new instance | Ensure `_memo_tool` / `_rl_tool` is module-level, not re-created |